In [ ]:
# Import libraries
from delta import DeltaTable
from pyspark.sql.functions import col, first, max, current_timestamp
import json

In [ ]:
# Define source schema and tables for fact_demand table in gold
source_schema = 'dbo'
source_table = 'generation_forecast'

In [ ]:
# Define parameters with None, as values will come from pipeline activities
# Source
source_lakehouse_path = None

# Target
target_lakehouse_path = None 
target_schema = None
target_table = None

# Dates
watermark_column = None
last_watermark = None

In [ ]:
# Define variables using values from parameters
source_path = f'{source_lakehouse_path}/Tables/{source_schema}/{source_table}'
target_path = f'{target_lakehouse_path}/Tables/{target_schema}/{target_table}'

# Load Helper Notebooks

Load helper notebook with the schema, merge keys and indicator mapping definition of gold tables inside "gold_objects" dictionary: 


In [ ]:
%run ./NB_Gold_Table_Schemas

In [ ]:
# Use gold_objects dictionary to define aditional variables
target_table_schema = gold_objects[f"{target_table}"]["schema"]
merge_keys = gold_objects[f"{target_table}"]["merge_keys"]

# Read and Transform New Data

In [ ]:
# Capture the execution timestamp to be used as the watermark boundary for Gold ingestion
# This ensures a consistent cutoff time and prevents missing rows if Silver is updated during execution
ingestion_timestamp = current_timestamp()

In [ ]:
# Read data older than last watermark
df = spark.read \
    .format('delta') \
    .load(source_path) \
    .filter(condition=col(watermark_column)>last_watermark) 

In [ ]:
# Add ingestion_timestamp as new column
df = df.withColumn("ingestion_timestamp", ingestion_timestamp)

In [ ]:
# Order by datetime_utc
df = df.orderBy('datetime_utc')

# Update Table

In [ ]:
# crear la tabla si no existe
try:
    delta_table = DeltaTable.forPath(spark, target_path)
    print('Table already exists')
except:
    delta_table = DeltaTable.createIfNotExists(spark) \
    .location(target_path) \
    .addColumns(target_table_schema) \
    .execute()
    print('Table has been created')

In [ ]:
# Define merge condition using merge_keys from helper notebook 
merge_condition = " AND ".join(f"t.{col} = s.{col}" for col in merge_keys) 

# Upsert
delta_table = DeltaTable.forPath(spark, target_path)
delta_table.alias("t").merge(
    df.alias("s"),
    merge_condition
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Statistics
metrics = delta_table.history(1).select("operationMetrics").collect()[0]["operationMetrics"]
rows_inserted = int(metrics["numTargetRowsInserted"])
rows_updated = int(metrics["numTargetRowsUpdated"])

In [ ]:
# Obtain max date in watermark_column to use as new watermark and store in a dictionary with the statistics
maxdate = df.select(max(watermark_column)).first()[0]

result = {
    "maxdate": maxdate.strftime("%Y-%m-%d %H:%M:%S") if maxdate else None,
    "rows_inserted": rows_inserted,
    "rows_updated": rows_updated
}

# Make the result values queriable for other activities in the pipeline
mssparkutils.notebook.exit(json.dumps(result))